In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd

folder = '/content/drive/MyDrive/depi_scraping/'
date = '2026-10-27'

offers = []
for site in ['dream2000', 'xprs', 'jumia']:
    with open(f'{folder}{site}_offers_{date}.json', encoding='utf-8') as f:
        offers += json.load(f)

df = pd.DataFrame(offers)
df.groupby(['site', 'category']).size()

Mounted at /content/drive


site       category
dream2000  phone       500
           tv           57
jumia      phone       478
           tv          426
xprs       phone       324
           tv           21
dtype: int64

In [ ]:
df.groupby(['site', 'category_site', 'category']).size()

site       category_site        category
dream2000  armadillo-tvs        tv            6
           ata-tvs              tv            6
           brands               tv           19
           fresh-tvs            tv            2
           haier-tvs            tv            8
           hisense-tvs          tv            4
           lg-tvs               tv           12
           mobiles              phone       500
jumia      /mobile-phones/      phone       478
           /televisions/        tv          426
xprs       32-inch-tv           tv            4
           43-inch-tv           tv            4
           55-inch-tv           tv            2
           65-inch-tv           tv            3
           mobile-phones-egypt  phone       324
           neo-qled-tvs         tv            3
           ultra-hd-4k-tvs      tv            5
dtype: int64

In [ ]:
category_map = df[['site', 'category_site', 'category']].drop_duplicates()
category_map = category_map.sort_values(['site', 'category_site'])
category_map.to_csv(folder + 'category_map.csv', index=False)
category_map

,site,category_site,category
551,dream2000,armadillo-tvs,tv
545,dream2000,ata-tvs,tv
500,dream2000,brands,tv
543,dream2000,fresh-tvs,tv
535,dream2000,haier-tvs,tv
531,dream2000,hisense-tvs,tv
519,dream2000,lg-tvs,tv
0,dream2000,mobiles,phone
902,jumia,/mobile-phones/,phone
1380,jumia,/televisions/,tv


In [ ]:
pd.set_option('display.max_rows', 200)

brands = df.groupby(['brand_raw', 'site']).size().unstack(fill_value=0)
brands = brands.sort_index(key=lambda s: s.str.lower())
brands

site,dream2000,jumia,xprs
brand_raw,,,
Apple,57,26,36
Arion,0,7,0
Armadillo,6,5,0
ATA,10,0,0
Ata,0,45,0
Barito,0,1,0
Beko,0,3,0
Bismarck,0,3,0
Caisson,0,1,0


In [ ]:
df['brand'] = df['brand_raw'].str.strip().str.lower()

print('before:', df['brand_raw'].nunique())
print('after :', df['brand'].nunique())

before: 86
after : 76


In [ ]:
aliases = {
    'infinx': 'infinix',
    'nokia egypt': 'nokia',
    'sky line': 'skyline',
    'redmi': 'xiaomi',
    'castle clothing': 'castle',
    'lg electronics': 'lg',
    'سامسونج': 'samsung',
    'ال جي': 'lg',
    'إل جي': 'lg',
    'شارب': 'sharp',
    'أبل': 'apple',
    'ابل': 'apple',
    'شاومي': 'xiaomi',
}

brand_aliases = pd.DataFrame(aliases.items(), columns=['alias', 'brand'])
brand_aliases.to_csv(folder + 'brand_aliases.csv', index=False, encoding='utf-8-sig')

df['brand'] = df['brand'].replace(aliases)
print('after aliases:', df['brand'].nunique())
brand_aliases

after aliases: 71


,alias,brand
0,infinx,infinix
1,nokia egypt,nokia
2,sky line,skyline
3,redmi,xiaomi
4,castle clothing,castle
5,lg electronics,lg
6,سامسونج,samsung
7,ال جي,lg
8,إل جي,lg
9,شارب,sharp


In [ ]:
import re

tv = df[df['category'] == 'tv'].copy()

CODE = re.compile(r"\b(?=[a-z0-9-]*[a-z])(?=(?:[a-z-]*\d){2})[a-z0-9-]{5,}\b", re.I)
NOT_CODE = re.compile(r"\d+(gb|tb|hp|inch|hz|p)?", re.I)

def code_of(row):
    text = f"{row['model_raw'] or ''} {row['title']}"
    codes = [c.replace('-', '').upper() for c in CODE.findall(text)]
    codes = [c for c in codes if not NOT_CODE.fullmatch(c)]
    return max(codes, key=len) if codes else None

tv['code'] = tv.apply(code_of, axis=1)

on2 = tv.groupby(['brand', 'code'])['site'].nunique()
print('codes on 2+ sites:', (on2 >= 2).sum())

codes on 2+ sites: 26


In [ ]:
examples = tv[tv['code'].str.contains('43U8000H|43LM6370|H55S80GUX', na=False)]
examples[['site', 'brand', 'code']].sort_values(['brand', 'code'])

,site,brand,code
1768,jumia,haier,EGYPTH55S80GUX
537,dream2000,haier,H55S80GUX
538,dream2000,haier,H55S80GUX
506,dream2000,lg,43LM6370PVA
1381,jumia,lg,43LM6370PVA
1493,jumia,lg,43LM6370PVA
1626,jumia,lg,MODEL43LM6370PVA
1470,jumia,lg,RECEIVER43LM6370PVA
886,xprs,samsung,43U8000H
1420,jumia,samsung,UA43U8000HUXEG


In [ ]:
rules = [
    ('*', 'prefix', 'MODEL'),
    ('*', 'prefix', 'RECEIVER'),
    ('*', 'prefix', 'WEBOS25'),
    ('*', 'prefix', 'WEBOS26'),
    ('*', 'prefix', 'EGYPT'),
    ('*', 'prefix', 'BLACK'),
    ('*', 'prefix', 'ANDROID'),
    ('samsung', 'suffix', 'AUXEG'),
    ('samsung', 'suffix', 'EUXEG'),
    ('samsung', 'suffix', 'UXEG'),
    ('samsung', 'suffix', 'EXEG'),
    ('samsung', 'prefix', 'UA'),
    ('samsung', 'prefix', 'QA'),
    ('fresh', 'suffix', '20976'),
    ('fresh', 'suffix', '20978'),
    ('ata', 'suffix', 'BLACK'),
    ('ata', 'prefix', 'ATA'),
]

model_suffixes = pd.DataFrame(rules, columns=['brand', 'position', 'text'])
model_suffixes.to_csv(folder + 'model_suffixes.csv', index=False)

def model_key(row):
    key = row['code']
    if not isinstance(key, str):
        return None
    for brand, position, text in rules:
        if brand not in ('*', row['brand']):
            continue
        if position == 'prefix' and key.startswith(text) and len(key) > len(text) + 4:
            key = key[len(text):]
        if position == 'suffix' and key.endswith(text) and len(key) > len(text) + 4:
            key = key[:-len(text)]
    return key

tv['model_key'] = tv.apply(model_key, axis=1)

on2 = tv.groupby(['brand', 'model_key'])['site'].nunique()
print('model keys on 2+ sites:', (on2 >= 2).sum())
model_suffixes

model keys on 2+ sites: 30


,brand,position,text
0,*,prefix,MODEL
1,*,prefix,RECEIVER
2,*,prefix,WEBOS25
3,*,prefix,WEBOS26
4,*,prefix,EGYPT
5,*,prefix,BLACK
6,*,prefix,ANDROID
7,samsung,suffix,AUXEG
8,samsung,suffix,EUXEG
9,samsung,suffix,UXEG
